In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import joblib
import json

In [2]:
# Load the dataset
data = pd.read_csv('D:/Agro_model/soil/dataset02/tea_nutrient_dataset.csv')

In [3]:
# Define input features (only sensor readings) and target variables
X_features = ['pH', 'moisture', 'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm']

y_targets = ['nitrogen_fertilizer_kg_ha', 'phosphorus_fertilizer_kg_ha', 
             'potassium_fertilizer_kg_ha', 'lime_application_kg_ha', 'irrigation_needed_mm']

X = data[X_features]
y = data[y_targets]

In [4]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# No categorical columns - only numeric columns
numeric_cols = ['pH', 'moisture', 'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm']

# Define preprocessing for numeric data (scaling)
numeric_transformer = StandardScaler()

In [6]:
# Create preprocessing steps (only numeric transformation)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols)
    ])

In [7]:
# Create a pipeline that includes preprocessing and a Random Forest model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', MultiOutputRegressor(RandomForestRegressor(
        n_estimators=100, 
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )))
])

In [8]:
# Train the model
print("Training the model...")
model.fit(X_train, y_train)

Training the model...


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['pH', 'moisture',
                                                   'nitrogen_ppm',
                                                   'phosphorus_ppm',
                                                   'potassium_ppm'])])),
                ('regressor',
                 MultiOutputRegressor(estimator=RandomForestRegressor(max_depth=15,
                                                                      min_samples_leaf=2,
                                                                      min_samples_split=5,
                                                                      random_state=42)))])

In [9]:
# Make predictions on test data
print("Evaluating the model...")
y_pred = model.predict(X_test)


Evaluating the model...


In [10]:
# Convert predictions to DataFrame for easier handling
y_pred_df = pd.DataFrame(y_pred, columns=y_targets)

In [12]:
# Evaluate the model
mae_scores = {}
r2_scores = {}
for i, col in enumerate(y_targets):
    mae = mean_absolute_error(y_test[col], y_pred_df[col])
    r2 = r2_score(y_test[col], y_pred_df[col])
    mae_scores[col] = mae
    r2_scores[col] = r2
    print(f"{col}: MAE = {mae:.2f}, R² = {r2:.4f}")

nitrogen_fertilizer_kg_ha: MAE = 6.26, R² = 0.9639
phosphorus_fertilizer_kg_ha: MAE = 0.54, R² = 0.9744
potassium_fertilizer_kg_ha: MAE = 2.43, R² = 0.9697
lime_application_kg_ha: MAE = 4.23, R² = 0.9718
irrigation_needed_mm: MAE = 0.64, R² = 0.9385


In [13]:
# Save the model
print("Saving the model...")
joblib.dump(model, 'tea_nutrient_recommendation_model.joblib')


Saving the model...


['tea_nutrient_recommendation_model.joblib']

In [15]:
# Create and save a metadata file with information about the model
model_metadata = {
    "model_version": "1.0.0",
    "input_features": X_features,
    "output_targets": y_targets,
    "numeric_features": numeric_cols,
    "performance_metrics": {
        "mae": {k: float(v) for k, v in mae_scores.items()},
        "r2": {k: float(v) for k, v in r2_scores.items()}
    },
    "optimal_ranges": {
        "pH": [4.5, 5.5],
        "moisture": [60, 80],
        "nitrogen_ppm": [150, 250],
        "phosphorus_ppm": [15, 35],
        "potassium_ppm": [80, 140]
    }
}

with open('tea_nutrient_model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

In [16]:
# Visualize feature importance
def plot_feature_importance():
    # Get feature names after preprocessing (only numeric features now)
    feature_names = numeric_cols
    
    # For each target, plot feature importance
    plt.figure(figsize=(15, 25))
    
    # Extract the Random Forest models from the MultiOutputRegressor
    rf_models = model.named_steps['regressor'].estimators_
    
    for i, (target, rf) in enumerate(zip(y_targets, rf_models)):
        # Get feature importances
        importances = rf.feature_importances_
        
        # Create a DataFrame for better visualization
        feature_importance = pd.DataFrame({
            'Feature': feature_names,
            'Importance': importances
        })
        
        # Sort by importance
        feature_importance = feature_importance.sort_values('Importance', ascending=False)
        
        # Plot
        plt.subplot(5, 1, i+1)
        plt.barh(feature_importance['Feature'][:10], feature_importance['Importance'][:10])
        plt.xlabel('Importance')
        plt.ylabel('Feature')
        plt.title(f'Top 10 Important Features for {target}')
        plt.tight_layout()
    
    plt.savefig('feature_importance.png')
    plt.close()

In [17]:
# Plot actual vs predicted values
def plot_actual_vs_predicted():
    plt.figure(figsize=(15, 20))
    
    for i, col in enumerate(y_targets):
        plt.subplot(5, 1, i+1)
        plt.scatter(y_test[col], y_pred_df[col], alpha=0.5)
        plt.plot([y_test[col].min(), y_test[col].max()], 
                [y_test[col].min(), y_test[col].max()], 'r--')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'Actual vs Predicted: {col}')
        
    plt.tight_layout()
    plt.savefig('actual_vs_predicted.png')
    plt.close()

In [18]:
# Create visualizations
print("Creating visualizations...")
plot_feature_importance()
plot_actual_vs_predicted()

Creating visualizations...


In [ ]:
# Create a function for making new predictions
def predict_fertilizer_needs(sensor_data):
    
    # Convert input to DataFrame
    input_df = pd.DataFrame([sensor_data])
    
    # Make prediction
    prediction = model.predict(input_df)
    
    # Convert to dictionary
    recommendation = {
        target: float(prediction[0][i])
        for i, target in enumerate(y_targets)
    }
    
    return recommendation

In [20]:
# Example usage of the prediction function
print("\nExample prediction:")
example_input = {
    'pH': 4.2,
    'moisture': 55.0,
    'nitrogen_ppm': 120.0,
    'phosphorus_ppm': 12.0,
    'potassium_ppm': 70.0
}

recommendation = predict_fertilizer_needs(example_input)
print(f"Sensor readings: {example_input}")
print(f"Recommendations:")
for nutrient, amount in recommendation.items():
    print(f"  {nutrient}: {amount:.2f}")


Example prediction:
Sensor readings: {'pH': 4.2, 'moisture': 55.0, 'nitrogen_ppm': 120.0, 'phosphorus_ppm': 12.0, 'potassium_ppm': 70.0}
Recommendations:
  nitrogen_fertilizer_kg_ha: 49.82
  phosphorus_fertilizer_kg_ha: 7.53
  potassium_fertilizer_kg_ha: 18.30
  lime_application_kg_ha: 80.17
  irrigation_needed_mm: 2.70
